# Question 3: Baseline U-Net Image Inpainting

**Problem Statement**: Dog image inpainting is an image restoration task where part of a dog image is masked and the model predicts the missing region. The goal is not only to copy nearby pixels, but to understand dog-specific visual structure, such as ears, fur texture, snout shape, eyes, body contours, and background consistency.

This notebook trains a plain U-Net-style reconstruction model for question 3: dog image inpainting. It is our project's baseline model. The goal is not to produce perfect image synthesis, but to show what a supervised CNN can learn from masked dog images using pixel-level reconstruction losses.

The U-Net baseline can reduce the reconstruction loss, but the generated missing regions may appear blurry or averaged. However, it can still be a useful baseline because it motivates us to explore stronger models such as edge-guided completion and pretrained diffusion inpainting.

## Pipeline

| Notebook | Role |
|---|---|
| `01_UNet` | baseline U-Net image inpainting trained from scratch |
| `02_SAM_Demo` | optional segmentation-based mask generation demo |
| `03_Edge_Generator` | edge-guided stage 1: predict missing structure |
| `04_Image_Completion` | edge-guided stage 2: reconstruct image content |
| `05_Demo_Stable_Diffusion` | advanced pretrained diffusion inpainting demo |
| `06_Full_Stable_Diffusion` | full SD pipeline |
| `00_Project_Comparison` | final side-by-side visual and quantitative comparison |

For fair comparison, use the same input images and the same synthetic masks across models whenever possible.

Environment Set Up:

In [ ]:
from google.colab import files
#uploaded = files.upload()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Data Loading

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Stanford Dogs dataset
!kaggle datasets download -d jessicali9530/stanford-dogs-dataset
!unzip -q -o stanford-dogs-dataset.zip -d stanford-dogs/
!echo "Done. Stanford Dogs folder structure:"
!ls stanford-dogs/

# Mixed Breed dataset
!kaggle datasets download -d anjalikapoor14/mixed-breed-dog-dataset
!unzip -q -o mixed-breed-dog-dataset.zip -d mixed-breed/
!echo "Done. Mixed Breed folder structure:"
!ls mixed-breed/

In [ ]:
from pathlib import Path
import os, math, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

DATA_DIR = Path('stanford-dogs/images/Images')
breed_folders = sorted([f for f in DATA_DIR.iterdir() if f.is_dir()])

records = []
for folder in breed_folders:
    # Folders look like 'n02085620-Chihuahua' — strip the prefix
    breed_name = folder.name.split('-', 1)[1].replace('_', ' ').title()
    for img_path in folder.glob('*.jpg'):
        records.append({
            'path':          str(img_path),
            'breed':         breed_name,
            'breed_folder':  folder.name,
        })

df = pd.DataFrame(records)
df.head()

**Train / validation / test split**

In [ ]:
from sklearn.model_selection import train_test_split

# 70 / 15 / 15 stratified split
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df['breed'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['breed'], random_state=42
)

print(f'Train:  {len(train_df):>6,}  ({len(train_df)/len(df):.0%})')
print(f'Val:    {len(val_df):>6,}  ({len(val_df)/len(df):.0%})')
print(f'Test:   {len(test_df):>6,}  ({len(test_df)/len(df):.0%})')
print(f'Total:  {len(df):>6,}')

train_df.to_csv('split_train.csv', index=False)
val_df.to_csv('split_val.csv', index=False)
test_df.to_csv('split_test.csv', index=False)


**Normalize: shift pixel values into the format the model expects**


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

to_tensor = transforms.ToTensor()
normalize = transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)

resized   = transforms.CenterCrop(224)(transforms.Resize(256)(demo_img))
tensor    = to_tensor(resized)             # shape [3, 224, 224], values 0–1
normed    = normalize(tensor)              # values centered around 0

# Helper: undo normalization for display
def denormalize(t, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    mean = torch.tensor(mean).view(3, 1, 1)
    std  = torch.tensor(std).view(3, 1, 1)
    return (t * std + mean).clamp(0, 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor('white')

axes[0].imshow(resized)
axes[0].set_title('Resized image\n(0–255 pixels)', color=NAVY)
axes[0].axis('off')

axes[1].imshow(normed.permute(1, 2, 0).numpy())
axes[1].set_title(f'After Normalize\n(values: {normed.min():.2f} to {normed.max():.2f})',
                  color=NAVY)
axes[1].axis('off')

axes[2].imshow(denormalize(normed).permute(1, 2, 0).numpy())
axes[2].set_title('De-normalized\n(back to looking normal)', color=NAVY)
axes[2].axis('off')

plt.tight_layout()
plt.show()
print(f'Tensor shape: {tuple(normed.shape)}')
print(f'Normalized pixel range: [{normed.min():.3f}, {normed.max():.3f}]')

**Wrap It In a PyTorch Dataset**

In [ ]:
import torchvision.transforms as transforms

IMG_SIZE = 128

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
])

eval_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

In [ ]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

class DogDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.classes = sorted(self.df['breed'].unique())
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = self.class_to_idx[row['breed']]
        return img, label

train_ds = DogDataset(train_df, transform=train_transforms)
val_ds   = DogDataset(val_df,   transform=eval_transforms)
test_ds  = DogDataset(test_df,  transform=eval_transforms)

# Sanity check
img, lbl = train_ds[0]
print(f'Number of breeds:  {len(train_ds.classes)}')
print(f'Train size:        {len(train_ds):,}')
print(f'Sample tensor:     shape={tuple(img.shape)}, dtype={img.dtype}')
print(f'Sample label:      {lbl}  ({train_ds.classes[lbl]})')

**DataLoaders: feed batches to the model**

In [ ]:
BATCH_SIZE  = 64
NUM_WORKERS = 2

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

In [ ]:
imgs, lbls = next(iter(train_loader))
print(f'Batch shape:        {tuple(imgs.shape)}    (batch, channels, height, width)')
print(f'Pixel value range:  [{imgs.min():.2f}, {imgs.max():.2f}]   (negatives are correct after normalization)')
print(f'Unique breeds:      {len(set(lbls.tolist()))} / {BATCH_SIZE}   (high diversity = shuffle working)')

# Modeling Preparation


In [ ]:
import os, random
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

**Set Image Folder Path:**

In [ ]:
print(os.listdir("/content"))

In [ ]:
SAVE_DIR = "/content/drive/MyDrive/Adv CV/CV Project Folder"
IMAGE_DIR = "/content/stanford-dogs"

Collect image path:

In [ ]:
image_paths = []

for_root = IMAGE_DIR
for root, dirs, files in os.walk(for_root):
    for file in files:
        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            image_paths.append(os.path.join(root, file))

print("Number of images:", len(image_paths))
print(image_paths[:3])

Define Transform:

In [ ]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

Create random mask function:

In [ ]:
def create_random_rect_mask(img, min_size=30, max_size=60):
    """
    img shape: [C, H, W]
    returns mask shape: [1, H, W]
    1 = visible area
    0 = missing area
    """
    _, H, W = img.shape
    mask = torch.ones((1, H, W))

    h = random.randint(min_size, max_size)
    w = random.randint(min_size, max_size)

    y = random.randint(0, H - h)
    x = random.randint(0, W - w)

    mask[:, y:y+h, x:x+w] = 0
    return mask

Prepare inpainting dataset:

* Output becomes:
| Variable       | Meaning                     |
| -------------- | --------------------------- |
| `model_input`  | 4-channel input to model    |
| `image`        | original ground truth image |
| `masked_image` | damaged image               |
| `mask`         | binary mask                 |


In [ ]:
class DogInpaintingDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self): # return total number of images
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        mask = create_random_rect_mask(image)

        masked_image = image * mask

        # Input has 4 channels: masked RGB image + mask
        model_input = torch.cat([masked_image, mask], dim=0)

        return model_input, image, masked_image, mask

Create train/val dataloaders:

In [ ]:
dataset = DogInpaintingDataset(image_paths, transform=transform)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))

Visualize original, masked, and mask images

In [ ]:
model_input, original, masked_image, mask = dataset[0]

plt.figure(figsize=(10, 3))

plt.subplot(1, 3, 1)
plt.imshow(original.permute(1, 2, 0))
plt.title("Original")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(masked_image.permute(1, 2, 0))
plt.title("Masked Image")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(mask.squeeze(), cmap="gray")
plt.title("Mask")
plt.axis("off")

plt.show()

# **U-Net Baseline**

Our baseline model uses a standard U-Net encoder–decoder architecture for dog image inpainting. The network reconstructs missing image regions by learning a mapping from masked dog images to completed outputs through convolutional downsampling and upsampling layers with skip connections that preserve spatial detail.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

**Define a small U-Net**

In [ ]:
class UNetSmall(nn.Module):
    def __init__(self):
        super().__init__()

        self.enc1 = nn.Sequential(
            nn.Conv2d(4, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU()
        )
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.ReLU()
        )
        self.pool2 = nn.MaxPool2d(2)

        self.bottleneck = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.ReLU()
        )

        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)

        self.dec2 = nn.Sequential(
            nn.Conv2d(256, 128, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.ReLU()
        )

        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)

        self.dec1 = nn.Sequential(
            nn.Conv2d(128, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU()
        )

        self.out = nn.Conv2d(64, 3, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        p1 = self.pool1(e1)

        e2 = self.enc2(p1)
        p2 = self.pool2(e2)

        b = self.bottleneck(p2)

        u2 = self.up2(b)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))

        u1 = self.up1(d2)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))

        return torch.sigmoid(self.out(d1))

### U-Net Model Architecture

In [ ]:
!pip install torchviz

In [ ]:
from torchviz import make_dot

model = UNetSmall()

x = torch.randn(1, 4, 128, 128)
y = model(x)

make_dot(y, params=dict(model.named_parameters()))

Initialize model

In [ ]:
base_UNet = UNetSmall().to(device)

criterion = nn.L1Loss()
optimizer = torch.optim.Adam(base_UNet.parameters(), lr=1e-3)

print(base_UNet)

## **Training baseline UNet Inpainting model**:

In [ ]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    train_loss = 0

    for model_input, original, masked_image, mask in train_loader:
        model_input = model_input.to(device)
        original = original.to(device)

        optimizer.zero_grad()

        output = model(model_input)
        loss = criterion(output, original)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss = 0

    with torch.no_grad():
        for model_input, original, masked_image, mask in val_loader:
            model_input = model_input.to(device)
            original = original.to(device)

            output = model(model_input)
            loss = criterion(output, original)

            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch [{epoch+1}/{num_epochs}] Train Loss: {avg_train_loss:.4f} Val Loss: {avg_val_loss:.4f}")

In [ ]:
# save model
torch.save(
    model.state_dict(),
    f"{SAVE_DIR}/base_unet_inpainting.pt"
)

print("Saved!")

Load the base UNet model:

In [ ]:
# load model

import torch

# recreate model architecture
base_UNet = UNetSmall()   # replace with your actual model class

# load weights
base_UNet.load_state_dict(
    torch.load(
        f"{SAVE_DIR}/models/base_unet_inpainting.pt",
        map_location=torch.device("cuda" if torch.cuda.is_available() else "cpu")
    )
)

# move to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_UNet = base_UNet.to(device)

# evaluation mode
base_UNet.eval()

print("Model loaded!")

Visualization:

In [ ]:
base_UNet.eval()

model_input, original, masked_image, mask = val_dataset[5]

with torch.no_grad():
    output = base_UNet(model_input.unsqueeze(0).to(device)).cpu().squeeze(0)

plt.figure(figsize=(12, 3))

plt.subplot(1, 4, 1)
plt.imshow(original.permute(1, 2, 0))
plt.title("Original")
plt.axis("off")

plt.subplot(1, 4, 2)
plt.imshow(masked_image.permute(1, 2, 0))
plt.title("Masked")
plt.axis("off")

plt.subplot(1, 4, 3)
plt.imshow(output.permute(1, 2, 0))
plt.title("U-Net Output")
plt.axis("off")

plt.subplot(1, 4, 4)
completed = masked_image + output * (1 - mask)
plt.imshow(completed.permute(1, 2, 0))
plt.title("Completed")
plt.axis("off")

plt.show()

As shown above, the model output is highly blurry and primarily reconstructs coarse color regions. While the overall shape is partially recovered, the model fails to capture fine-grained details such as fur texture, facial structure, and clear body boundaries of the dog.

### Evaluation:

In [ ]:
# Load baseline U-Net
base_UNet = UNetSmall().to(device)

base_UNet.load_state_dict(
    torch.load(f"{SAVE_DIR}/models/base_unet_inpainting.pt",
    map_location=device)
)

base_UNet.eval()

In [ ]:
def psnr(mse):
    return 20 * torch.log10(torch.tensor(1.0)) - 10 * torch.log10(torch.tensor(mse))

@torch.no_grad()
def evaluate_inpainting_model(model, loader, device, max_batches=None):
    base_UNet.eval()

    total_l1 = 0
    total_mse = 0
    total_psnr = 0
    n = 0

    for batch_idx, batch in enumerate(loader):
        model_input, image, masked_image, mask = batch

        model_input = model_input.to(device)
        image = image.to(device)
        masked_image = masked_image.to(device)
        mask = mask.to(device)

        pred = model(model_input)
        pred = torch.clamp(pred, 0, 1)

        # completed image keeps known pixels and fills missing pixels with prediction
        completed = masked_image + pred * (1 - mask)

        l1 = F.l1_loss(completed, image, reduction="mean")
        mse = F.mse_loss(completed, image, reduction="mean")
        batch_psnr = psnr(mse.item())

        batch_size = image.size(0)
        total_l1 += l1.item() * batch_size
        total_mse += mse.item() * batch_size
        total_psnr += batch_psnr.item() * batch_size
        n += batch_size

        if max_batches is not None and batch_idx + 1 >= max_batches:
            break

    return {
        "L1 Loss": total_l1 / n,
        "MSE": total_mse / n,
        "PSNR": total_psnr / n
    }

metrics = evaluate_inpainting_model(base_UNet, val_loader, device)
metrics

For visual comparison:

In [ ]:
@torch.no_grad()
def show_inpainting_results(model, loader, device, num_images=5):
    model.eval()

    model_input, image, masked_image, mask = next(iter(loader))

    model_input = model_input.to(device)
    image = image.to(device)
    masked_image = masked_image.to(device)
    mask = mask.to(device)

    pred = model(model_input)
    pred = torch.clamp(pred, 0, 1)

    completed = masked_image + pred * (1 - mask)

    image = image.cpu()
    masked_image = masked_image.cpu()
    pred = pred.cpu()
    completed = completed.cpu()

    num_images = min(num_images, image.size(0))

    fig, axes = plt.subplots(num_images, 4, figsize=(12, 3 * num_images))

    for i in range(num_images):
        imgs = [image[i], masked_image[i], pred[i], completed[i]]
        titles = ["Original", "Masked Image", "U-Net Prediction", "Completed Image"]

        for j in range(4):
            ax = axes[i, j] if num_images > 1 else axes[j]
            ax.imshow(imgs[j].permute(1, 2, 0))
            ax.set_title(titles[j])
            ax.axis("off")

    plt.tight_layout()
    plt.show()

show_inpainting_results(base_UNet, val_loader, device, num_images=5)

Although our base UNet model predictions are semantically reasonable, and it is not outputting garbage/noise, this initial training pipeline works. However, from our evaluation, we found 2 main issues that we wanted to solve in the next step:

1. The mask is not consistently on the dog in the image
2. The outputs are quite blurry

## Improving the base UNet model

#### 1. Improve the random masking issue: By adding a new center-biased mask


In [ ]:
import random
import torch
from torch.utils.data import Dataset
from PIL import Image

def create_center_biased_mask(image, min_size=0.30, max_size=0.50, num_masks=1):
    _, h, w = image.shape
    mask = torch.ones((1, h, w))

    for _ in range(num_masks):
        mask_h = random.randint(int(h * min_size), int(h * max_size))
        mask_w = random.randint(int(w * min_size), int(w * max_size))

        center_y = random.randint(int(h * 0.35), int(h * 0.65))
        center_x = random.randint(int(w * 0.35), int(w * 0.65))

        y1 = max(0, center_y - mask_h // 2)
        x1 = max(0, center_x - mask_w // 2)

        y2 = min(h, y1 + mask_h)
        x2 = min(w, x1 + mask_w)

        mask[:, y1:y2, x1:x2] = 0

    return mask


class DogInpaintingDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        # new center-biased mask
        mask = create_center_biased_mask(
            image,
            min_size=0.25,
            max_size=0.45,
            num_masks=1
        )

        masked_image = image * mask

        # input has 4 channels: masked RGB image + mask
        model_input = torch.cat([masked_image, mask], dim=0)

        return model_input, image, masked_image, mask

In [ ]:
dataset = DogInpaintingDataset(image_paths, transform=transform)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))

In [ ]:
model_input, image, masked_image, mask = next(iter(train_loader))

show_batch = min(5, image.size(0))

fig, axes = plt.subplots(show_batch, 3, figsize=(9, 3 * show_batch))

for i in range(show_batch):
    axes[i, 0].imshow(image[i].permute(1, 2, 0))
    axes[i, 0].set_title("Original")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(masked_image[i].permute(1, 2, 0))
    axes[i, 1].set_title("Center-Biased Masked")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(mask[i].squeeze(), cmap="gray", vmin=0, vmax=1)
    axes[i, 2].set_title("Mask")
    axes[i, 2].axis("off")

plt.tight_layout()
plt.show()

Now the masks are all centered around the dog.

Recreate datasets & dataloaders:


In [ ]:
from torch.utils.data import DataLoader, random_split

# Rebuild dataset using the new center-biased DogInpaintingDataset
dataset = DogInpaintingDataset(image_paths, transform=transform)

# Same 80/20 split as before
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))

#### Create a new UNet model

In [ ]:
center_UNet = UNetSmall().to(device)

Training the new UNet Model:

In [ ]:
center_UNet = UNetSmall().to(device)

criterion = nn.L1Loss()
optimizer = torch.optim.Adam(center_UNet.parameters(), lr=1e-4)

In [ ]:
num_epochs = 20

for epoch in range(num_epochs):
    center_UNet.train()
    train_loss = 0

    for model_input, original, masked_image, mask in train_loader:
        model_input = model_input.to(device)
        original = original.to(device)

        optimizer.zero_grad()

        output = center_UNet(model_input)
        loss = criterion(output, original)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    center_UNet.eval()
    val_loss = 0

    with torch.no_grad():
        for model_input, original, masked_image, mask in val_loader:
            model_input = model_input.to(device)
            original = original.to(device)

            output = center_UNet(model_input)
            loss = criterion(output, original)

            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch [{epoch+1}/{num_epochs}] Train Loss: {avg_train_loss:.4f} Val Loss: {avg_val_loss:.4f}")

In [ ]:
config = {
    "epochs": num_epochs,
    "batch_size": 32,
    "learning_rate": 1e-4,
    "loss": "L1Loss",
    "mask_type": "center_biased",
    "model": "UNetSmall"
}

torch.save(
    config,
    f"{SAVE_DIR}/center_unet_config.pt"
)

Save the model:

In [ ]:
torch.save(
    center_UNet.state_dict(),
    f"{SAVE_DIR}/center_mask_unet_inpainting.pt"
)

In [ ]:
torch.save(
    center_UNet.state_dict(),
    f"{SAVE_DIR}/center_mask_unet_inpainting.pt"
)

Evaluation:

In [ ]:
metrics = evaluate_inpainting_model(center_UNet, val_loader, device)
metrics

In [ ]:
metrics = evaluate_inpainting_model(center_UNet, val_loader, device)

print(metrics)

torch.save(
    metrics,
    f"{SAVE_DIR}/center_unet_metrics.pt"
)

In [ ]:
show_inpainting_results(center_UNet, val_loader, device, num_images=5)

In [ ]:
import os
from torchvision.utils import save_image
import torch

@torch.no_grad()
def save_inpainting_examples(model, loader, device, save_dir, num_images=8):
    model.eval()

    os.makedirs(save_dir, exist_ok=True)

    model_input, image, masked_image, mask = next(iter(loader))

    model_input = model_input.to(device)
    image = image.to(device)
    masked_image = masked_image.to(device)
    mask = mask.to(device)

    pred = model(model_input)
    pred = torch.clamp(pred, 0, 1)

    completed = masked_image + pred * (1 - mask)

    image = image.cpu()
    masked_image = masked_image.cpu()
    pred = pred.cpu()
    completed = completed.cpu()

    num_images = min(num_images, image.size(0))

    for i in range(num_images):
        save_image(image[i], f"{save_dir}/example_{i}_original.png")
        save_image(masked_image[i], f"{save_dir}/example_{i}_masked.png")
        save_image(pred[i], f"{save_dir}/example_{i}_prediction.png")
        save_image(completed[i], f"{save_dir}/example_{i}_completed.png")

    print(f"Saved {num_images} examples to {save_dir}")

In [ ]:
save_inpainting_examples(
    center_UNet,
    val_loader,
    device,
    f"{SAVE_DIR}/center_unet_examples",
    num_images=10
)